# Sheet 3 — P, L, R and the dual graph (Chapter 4)

Triads sharing two notes are adjacent. The graph on the 24 triads with P/L/R edges is the
*dual* of the Tonnetz triangulation (Douthett & Steinbach's "chicken-wire torus").

In [1]:
# Setup: make the repository importable whether or not `pip install -e .` was run
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, networkx as nx, matplotlib.pyplot as plt
from tonnetzkit import *
%matplotlib inline

In [2]:
G = plr_graph()
print('cubic?', set(dict(G.degree()).values()), ' bipartite?', nx.is_bipartite(G), ' diameter:', nx.diameter(G))
import collections
D = plr_distance_table(); Tr = all_triads()
print('distance distribution over ordered pairs:', collections.Counter(D[a][b] for a in Tr for b in Tr if a != b))

cubic? {3}  bipartite? True  diameter: 5
distance distribution over ordered pairs: Counter({3: 192, 2: 144, 4: 120, 1: 72, 5: 24})


### Famous cycles: hexagons, hexatonic and octatonic systems

In [3]:
C = Triad.parse('C')
print('hexagon around note C  (PLR)^2 :', apply_word(C, 'PLRPLR'))
print('hexatonic cycle        (PL)^3  :', apply_word(C, 'PLPLPL'))
print('octatonic cycle        (PR)^4  :', apply_word(C, 'PRPRPRPR'))
print('all triads containing C:', [t for t in all_triads() if 0 in t.pcs])

hexagon around note C  (PLR)^2 : [C, Cm, G#, Fm, F, Am, C]
hexatonic cycle        (PL)^3  : [C, Cm, G#, G#m, E, Em, C]
octatonic cycle        (PR)^4  : [C, Cm, D#, D#m, F#, F#m, A, Am, C]
all triads containing C: [C, F, G#, Cm, Fm, Am]


### A correction worth making
The original slide lists "going around a hexagon" as C → Am → F → Dm → G → Em → C. Let us test every step:

In [4]:
seq = [Triad.parse(s) for s in 'C Am F Dm G Em C'.split()]
for a, b in zip(seq, seq[1:]):
    print(f'{a!s:>3} -> {b!s:<3}', 'adjacent' if G.has_edge(a, b) else f'NOT adjacent (distance {D[a][b]})')

  C -> Am  adjacent
 Am -> F   adjacent
  F -> Dm  adjacent
 Dm -> G   NOT adjacent (distance 3)
  G -> Em  adjacent
 Em -> C   adjacent


Dm → G shares only one note, so this is a *diatonic* chain of thirds, not a face of the dual graph. The true hexagon around C is C–Cm–A♭–Fm–F–Am.

### The PLR group

In [5]:
idx = {t: i for i, t in enumerate(Tr)}
gens = [tuple(idx[f(t)] for t in Tr) for f in (P, L, R)]
grp = {tuple(range(24))}; frontier = list(grp)
while frontier:
    nxt = []
    for g in frontier:
        for h in gens:
            c = tuple(h[g[i]] for i in range(24))
            if c not in grp: grp.add(c); nxt.append(c)
    frontier = nxt
print('order of <P,L,R> =', len(grp), ' (dihedral of order 24)')
print('P commutes with transposition:', all(P(transpose(t, 5)) == transpose(P(t), 5) for t in Tr))

order of <P,L,R> = 24  (dihedral of order 24)
P commutes with transposition: True


## Exercises
1. Show that the voice-leading cost (total semitones moved) is 1 for P and L and 2 for R.
2. Find a shortest path from C major to F♯ major. Is it unique?
3. Prove (or check) that the PLR graph is bipartite. Musically, what are the two parts?
<details><summary><b>Show solution</b></summary>

```python
for op in (P, L, R): print(op.__name__, {voice_leading_distance(t, op(t)) for t in Tr})
paths = list(nx.all_shortest_paths(G, Triad.parse('C'), Triad.parse('F#')))
print(len(paths), 'shortest paths of length', len(paths[0])-1); print(paths[0])
```
3. Every P, L, R swaps major and minor, so {majors} and {minors} are the two parts — `nx.is_bipartite(G)` is True.

</details>